# 🧑‍💼 Adult Income — Classification Analysis
**Dataset**: [UCI Adult / Census Income](https://archive.ics.uci.edu/dataset/2/adult)  
**Goal**: Predict whether a person's annual income exceeds $50K based on census attributes (age, education, occupation, hours worked, etc.)

---

## 🗺️ Notebook Roadmap
| Section | What You'll Learn |
|---|---|
| 0. Setup | Libraries for classification (SVC, Random Forest, XGBoost, LightGBM) |
| 1. EDA | Understanding the census data and the target class balance |
| 2. Assumptions Validation | What each algorithm needs to work well |
| 3. Train/Test Split | Stratified splitting for classification |
| 4. Cross-Validation | Comparing 4 algorithms reliably |
| 5. Hyperparameter Tuning | Tuning each of the 4 algorithms |
| 6. Results Validation | Confusion matrices, ROC curves, feature importance |

---

### 🔑 Key Concept: What is Classification?
> **Classification** is supervised learning where the goal is to predict a **category/label** rather than a continuous number.  
> Here, the label is binary: does this person earn **`>50K`** or **`<=50K`** per year? This is called **binary classification**.

---
## 0. Environment Setup

### 📦 New Libraries for This Notebook
- `xgboost` — Gradient boosting library, very fast and accurate, widely used in industry
- `lightgbm` — Microsoft's gradient boosting library, optimized for speed on large datasets
- `sklearn.svm.SVC` — Support Vector Classifier
- `sklearn.ensemble.RandomForestClassifier` — Ensemble of decision trees for classification


In [ ]:
# ============================================================
# CELL 0-A: Install all required packages
# ============================================================
# Run this once, then comment it out.
# !pip install ucimlrepo pandas numpy matplotlib seaborn scikit-learn scipy statsmodels xgboost lightgbm

In [ ]:
# ============================================================
# CELL 0-B: Import all libraries
# ============================================================

# --- Data loading ---
from ucimlrepo import fetch_ucirepo

# --- Data manipulation ---
import pandas as pd
import numpy as np
import textwrap

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Statistics ---
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- Preprocessing & model selection ---
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Models ---
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# --- Metrics ---
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)

# --- Utility ---
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All imports successful!")

---
## 📖 Feature Dictionary

Just like in the regression notebook, let's build a lookup dictionary so every column name is self-explanatory throughout the analysis. Source: [UCI Adult dataset documentation](https://archive.ics.uci.edu/dataset/2/adult).

In [ ]:
# ============================================================
# CELL 0-C: Feature dictionary — maps column names to descriptions
# ============================================================

FEATURE_DESCRIPTIONS = {
    'age': 'Age of the individual (years)',
    'workclass': 'Type of employer (Private, Government, Self-employed, etc.)',
    'fnlwgt': 'Census final sampling weight — a statistical weighting factor, not a personal trait',
    'education': 'Highest level of education completed',
    'education-num': 'Education level encoded as a number (higher = more education)',
    'marital-status': 'Marital status (Married, Divorced, Never-married, etc.)',
    'occupation': 'Type of job (Tech-support, Sales, Exec-managerial, etc.)',
    'relationship': "Person's relationship role within their household",
    'race': 'Race of the individual',
    'sex': 'Sex of the individual',
    'capital-gain': 'Income from investments/capital gains (USD)',
    'capital-loss': 'Losses from investments/capital losses (USD)',
    'hours-per-week': 'Number of hours worked per week',
    'native-country': 'Country of origin',
    'income': 'Whether annual income exceeds $50K (TARGET): <=50K or >50K',
}


def describe(feature_name):
    """Return a plain-English description for a single feature name."""
    return FEATURE_DESCRIPTIONS.get(feature_name.strip(), 'No description available')


def describe_features(feature_names):
    """Return a DataFrame of feature + description for display alongside other tables."""
    return pd.DataFrame({
        'Feature': feature_names,
        'Description': [describe(f) for f in feature_names]
    })


print(f"✅ Feature dictionary loaded: {len(FEATURE_DESCRIPTIONS)} entries")
feature_dict_df = describe_features([f for f in FEATURE_DESCRIPTIONS.keys() if f != 'income'])
print("\n📖 Full feature dictionary:")
feature_dict_df

---
## 1. Exploratory Data Analysis (EDA)

Same goals as before: understand shape, types, missing values, distributions, outliers, and relationships with the target — but this time the target is a **category**, not a number.

In [ ]:
# ============================================================
# CELL 1-A: Load the dataset
# ============================================================
print("Downloading dataset from UCI...")
dataset = fetch_ucirepo(id=2)

X = dataset.data.features
y = dataset.data.targets

# Strip whitespace from column names (same quirk as the regression dataset)
X.columns = X.columns.str.strip()
y.columns = y.columns.str.strip()

df = pd.concat([X, y], axis=1)
df.columns = df.columns.str.strip()

# Defensive fix: this dataset can also have leading/trailing whitespace inside
# STRING VALUES themselves (e.g., ' Private' instead of 'Private', or ' ?' instead
# of '?'), depending on how ucimlrepo serves it. If we don't strip these, the
# '?' -> NaN replacement below could silently miss missing values entirely.
# Strip whitespace from every text column to guard against this.
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# The 'income' column may contain values like '<=50K', '>50K', '<=50K.', '>50K.'
# (the trailing period comes from how UCI stored the test split). Clean these up.
df['income'] = df['income'].str.strip().str.replace('.', '', regex=False)

# Replace '?' (used as a missing value marker in this dataset) with proper NaN
df = df.replace('?', np.nan)

print(f"✅ Dataset loaded!")
print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Target classes: {df['income'].unique().tolist()}")

In [ ]:
# ============================================================
# CELL 1-B: First look at the data
# ============================================================
print("📋 First 5 rows of the dataset:")
df.head()

In [ ]:
# ============================================================
# CELL 1-C: Data types and non-null counts
# ============================================================
print("📊 Dataset Info:")
df.info()

In [ ]:
# ============================================================
# CELL 1-D: Statistical summary (numeric columns)
# ============================================================
print("📈 Statistical Summary (numeric columns):")
df.describe().T.round(2)

In [ ]:
# ============================================================
# CELL 1-E: Missing values check
# ============================================================
# This dataset uses '?' for missing values, which we already converted to NaN.
# 3 columns commonly have missing values: workclass, occupation, native-country.

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values found!")
else:
    print(f"⚠️ Found missing values in {len(missing_df)} columns:")
    print(missing_df)
    print(f"\nTotal rows affected: {df.isnull().any(axis=1).sum():,} out of {len(df):,}")
    print("💡 Strategy: drop rows with missing values (they're a small % of the dataset)")

In [ ]:
# ============================================================
# CELL 1-F: Target variable analysis — 'income'
# ============================================================
# WHAT IS CLASS BALANCE?
# Class balance refers to how evenly the target categories are represented.
# A 'balanced' dataset has roughly equal counts of each class.
# An 'imbalanced' dataset has one class much more common than the other.
#
# WHY IT MATTERS:
# If 75% of people earn <=50K, a model that ALWAYS predicts '<=50K' would be
# 75% accurate without learning anything useful! This is why we need metrics
# beyond accuracy (precision, recall, F1, ROC-AUC) for imbalanced data.

class_counts = df['income'].value_counts()
class_pct = df['income'].value_counts(normalize=True) * 100

print("🎯 Target Variable: 'income'")
for cls, count in class_counts.items():
    print(f"   {cls:<8} {count:>7,} rows  ({class_pct[cls]:.1f}%)")

plt.figure(figsize=(7, 5))
colors = ['#1f77b4', '#d62728']
bars = plt.bar(class_counts.index, class_counts.values, color=colors, alpha=0.85, edgecolor='white')
for bar, pct in zip(bars, class_pct.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300, f'{pct:.1f}%',
             ha='center', fontsize=12, fontweight='bold')
plt.title('Target Class Distribution: income', fontsize=13, fontweight='bold')
plt.ylabel('Number of People')
plt.tight_layout()
plt.show()

print("\n💡 This dataset is MODERATELY IMBALANCED (~76% / ~24%).")
print("   We'll use stratified splitting and look at precision/recall/F1/ROC-AUC,")
print("   not just accuracy, when evaluating models.")

In [ ]:
# ============================================================
# CELL 1-G: Numeric feature distributions
# ============================================================
numeric_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    axes[i].hist(df[feat].dropna(), bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    wrapped_desc = '\n'.join(textwrap.wrap(describe(feat), width=32))
    axes[i].text(0.5, 1.15, wrapped_desc, transform=axes[i].transAxes,
                 ha='center', va='bottom', fontsize=8, color='gray', style='italic')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    skew_val = df[feat].skew()
    axes[i].text(0.97, 0.95, f'skew={skew_val:.1f}', transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=9, color='red')

plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("💡 'capital-gain' and 'capital-loss' are extremely skewed — most people report $0.")
print("   'fnlwgt' is a census weighting factor, not a personal attribute — limited predictive value.")

print("\n📋 Feature reference:")
print(describe_features(numeric_features).to_string(index=False))

In [ ]:
# ============================================================
# CELL 1-H: Categorical feature distributions
# ============================================================
categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex']

fig, axes = plt.subplots(4, 2, figsize=(16, 18))
axes = axes.flatten()

for i, feat in enumerate(categorical_features):
    value_counts = df[feat].value_counts().head(10)
    axes[i].barh(value_counts.index[::-1], value_counts.values[::-1], color='steelblue', alpha=0.85)
    axes[i].set_title(f"{feat}\n({describe(feat)})", fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Count')

axes[-1].axis('off')  # hide unused 8th subplot

plt.suptitle('Categorical Feature Distributions (Top 10 categories each)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 1-I: Outlier detection — numeric features
# ============================================================
# Same IQR method as the regression notebook.

def count_outliers_iqr(series):
    s = series.dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return ((s < lower) | (s > upper)).sum()

outlier_data = []
for col in numeric_features:
    n_out = count_outliers_iqr(df[col])
    pct = n_out / len(df) * 100
    outlier_data.append({'Feature': col, 'Outlier Count': n_out, 'Outlier %': round(pct, 2)})

outlier_df = pd.DataFrame(outlier_data).sort_values('Outlier %', ascending=False)
print("🔍 Outlier counts (IQR method):")
print(outlier_df.to_string(index=False))

fig, axes = plt.subplots(1, len(numeric_features), figsize=(18, 5))
for i, feat in enumerate(numeric_features):
    axes[i].boxplot(df[feat].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=sns.color_palette('muted')[i % 6], alpha=0.7),
                    medianprops=dict(color='black', linewidth=2))
    axes[i].set_title(feat, fontsize=9)

plt.suptitle('Box Plots — Outlier Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 'capital-gain' and 'capital-loss' show massive outlier rates — this is expected since")
print("   the vast majority of people report $0, while a few report large amounts.")
print("   Tree-based models (RF, XGBoost, LightGBM) handle this naturally without transformation.")

In [ ]:
# ============================================================
# CELL 1-J: Correlation heatmap — numeric features + target
# ============================================================
# To correlate with the target, we need 'income' as a number.
# We create a temporary binary encoding: 0 = '<=50K', 1 = '>50K'

df_corr = df[numeric_features].copy()
df_corr['income_binary'] = (df['income'] == '>50K').astype(int)

corr_with_target = df_corr.corr()['income_binary'].drop('income_binary')
corr_sorted = corr_with_target.abs().sort_values(ascending=False)

print("🔗 Numeric features correlated with income (>50K = 1):\n")
for feat in corr_sorted.index:
    corr = corr_with_target[feat]
    bar = '█' * int(abs(corr) * 60)
    direction = '+' if corr > 0 else '-'
    print(f"  {direction} {feat:<18} {corr:+.4f}  {bar}")
    print(f"      \u2514\u2500 {describe(feat)}")

plt.figure(figsize=(8, 6))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features + Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 1-K: Income vs categorical features
# ============================================================
# How does the proportion of >50K earners change across categories?

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

plot_features = ['education', 'marital-status', 'sex', 'occupation']

for i, feat in enumerate(plot_features):
    # % of each category that earns >50K
    pct_above = df.groupby(feat)['income'].apply(lambda x: (x == '>50K').mean() * 100)
    pct_above = pct_above.sort_values(ascending=True)

    axes[i].barh(pct_above.index, pct_above.values, color='seagreen', alpha=0.85)
    axes[i].set_title(f"% Earning >50K by {feat}", fontsize=11, fontweight='bold')
    axes[i].set_xlabel('% earning >50K')
    axes[i].tick_params(axis='y', labelsize=8)

plt.suptitle('Income Rate by Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Higher education, being married, and certain occupations strongly correlate with >50K income.")

---
## 2. Assumptions Validation

### 🔑 Do Classification Algorithms Have Assumptions?
> Unlike linear regression, tree-based models (Random Forest, XGBoost, LightGBM) make **very few statistical assumptions** — they don't require normally-distributed residuals or linear relationships.  
> However, **SVC (Support Vector Classifier)** is sensitive to feature scale, and **all models** are affected by class imbalance and multicollinearity in different ways.

**What we check here:**

| # | Check | Why it matters |
|---|---|---|
| 1 | Class Imbalance | Affects metric choice and possibly `class_weight` settings |
| 2 | Multicollinearity (VIF) | Affects SVC's linear kernel and coefficient interpretation |
| 3 | Feature Scaling Needs | SVC requires scaled features; tree models don't |
| 4 | Categorical Cardinality | High-cardinality categories affect one-hot encoding size |
| 5 | Missing Data Strategy | Confirmed approach for handling '?' values |

In [ ]:
# ============================================================
# CELL 2-A: Drop rows with missing values
# ============================================================
# We drop rows with missing workclass/occupation/native-country.
# This is a small % of the data (~7%), and dropping is simpler and safer
# than imputing categorical values for a learning exercise.

rows_before = len(df)
df_clean = df.dropna().reset_index(drop=True)
rows_after = len(df_clean)

print(f"Rows before dropping missing: {rows_before:,}")
print(f"Rows after dropping missing:  {rows_after:,}")
print(f"Rows dropped:                 {rows_before - rows_after:,} ({(rows_before-rows_after)/rows_before*100:.1f}%)")

In [ ]:
# ============================================================
# CELL 2-B: Assumption 1 — Class Imbalance (revisited)
# ============================================================
class_counts_clean = df_clean['income'].value_counts()
class_pct_clean = df_clean['income'].value_counts(normalize=True) * 100

print("🔬 Class balance after cleaning:")
for cls, count in class_counts_clean.items():
    print(f"   {cls:<8} {count:>7,} rows  ({class_pct_clean[cls]:.1f}%)")

imbalance_ratio = class_counts_clean.max() / class_counts_clean.min()
print(f"\n   Imbalance ratio: {imbalance_ratio:.2f} : 1")
if imbalance_ratio > 3:
    print("   ⚠️  Notable imbalance — use stratified splits, and consider class_weight='balanced'")
    print("   ⚠️  Don't rely on accuracy alone — track precision, recall, F1, ROC-AUC")
else:
    print("   ✅ Roughly balanced — accuracy is a reasonable metric")

In [ ]:
# ============================================================
# CELL 2-C: Assumption 2 — Multicollinearity (VIF) on numeric features
# ============================================================
# Same VIF interpretation as the regression notebook:
#   VIF = 1-5   → OK
#   VIF > 5     → Moderate concern
#   VIF > 10    → Severe concern

X_numeric = df_clean[numeric_features].copy()
X_numeric_scaled = StandardScaler().fit_transform(X_numeric)

vif_data = []
for i, col in enumerate(numeric_features):
    vif = variance_inflation_factor(X_numeric_scaled, i)
    vif_data.append({'Feature': col, 'VIF': round(vif, 2), 'Description': describe(col)})

vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
print("🔬 VIF for numeric features:")
print(vif_df.to_string(index=False))

print("\n💡 'education-num' and 'education' carry duplicate information (one is a numeric")
print("   encoding of the other). We'll keep 'education' (more interpretable for one-hot encoding)")
print("   and drop 'education-num' to avoid redundancy.")

In [ ]:
# ============================================================
# CELL 2-D: Assumption 3 — Feature Scaling Needs
# ============================================================
# SVC computes distances between points — features on different scales
# (e.g., 'age' 17-90 vs 'capital-gain' 0-99999) would let large-magnitude
# features dominate the decision boundary.
#
# Random Forest, XGBoost, LightGBM split on individual feature thresholds —
# scale doesn't affect their splits, so scaling is OPTIONAL for them but
# REQUIRED for SVC.

print("📏 Numeric feature ranges (before scaling):")
ranges_df = df_clean[numeric_features].agg(['min', 'max', 'mean', 'std']).T.round(2)
print(ranges_df)

print("\n💡 Ranges vary by orders of magnitude (e.g., 'age' vs 'capital-gain').")
print("   → We'll use a single preprocessing Pipeline with StandardScaler for ALL models.")
print("   → This keeps the pipeline consistent and doesn't hurt tree-based models.")

In [ ]:
# ============================================================
# CELL 2-E: Assumption 4 — Categorical Cardinality
# ============================================================
# 'Cardinality' = number of unique categories in a column.
# High-cardinality categorical features create MANY columns when one-hot encoded,
# which can slow down training and increase the risk of overfitting (especially for SVC).

categorical_cols_final = ['workclass', 'education', 'marital-status', 'occupation',
                           'relationship', 'race', 'sex', 'native-country']

cardinality = df_clean[categorical_cols_final].nunique().sort_values(ascending=False)
print("🔬 Categorical feature cardinality:")
for feat, n in cardinality.items():
    print(f"   {feat:<18} {n:>3} unique values   ({describe(feat)})")

total_ohe_cols = cardinality.sum()
print(f"\n   Total one-hot encoded columns from categoricals: ~{total_ohe_cols}")
print("   💡 'native-country' has the highest cardinality (mostly 'United-States').")
print("      We'll keep it — tree models handle high-cardinality one-hot features fine,")
print("      and OneHotEncoder(handle_unknown='ignore') protects against unseen categories.")

---
## 3. Training & Testing Dataset Selection

### 🔑 Why Stratified Splitting for Classification?
> **Stratified splitting** ensures both the train and test sets have the **same proportion of each class** as the full dataset.  
> Without it, a random split could (by chance) put too many `>50K` examples in the test set, making evaluation misleading.

This is the key difference from the regression notebook — for regression there's no "class" to preserve, but for classification, preserving class proportions is standard practice.

In [ ]:
# ============================================================
# CELL 3-A: Prepare final feature matrix and target
# ============================================================

# Drop 'education-num' (redundant with 'education') and 'fnlwgt' (sampling weight, not predictive)
DROP_COLS = ['education-num', 'fnlwgt']

FEATURES = [c for c in df_clean.columns if c not in DROP_COLS + ['income']]
X_final = df_clean[FEATURES].copy()

# Encode target: 0 = '<=50K', 1 = '>50K'
y_final = (df_clean['income'] == '>50K').astype(int)

NUMERIC_FEATURES = [c for c in numeric_features if c not in DROP_COLS]
CATEGORICAL_FEATURES = categorical_cols_final

print(f"Feature matrix shape: {X_final.shape}")
print(f"Target vector shape:  {y_final.shape}")
print(f"\nNumeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")
print(f"\nTarget encoding: 0 = '<=50K', 1 = '>50K'")

In [ ]:
# ============================================================
# CELL 3-B: Stratified train/test split
# ============================================================
# stratify=y_final ensures both splits preserve the ~76% / ~24% class ratio.

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    stratify=y_final,
    random_state=RANDOM_STATE
)

print("✅ Stratified data split complete!")
print(f"\nTraining set: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Test set:     X_test  {X_test.shape}, y_test  {y_test.shape}")

In [ ]:
# ============================================================
# CELL 3-C: Verify class balance is preserved
# ============================================================

train_pct = y_train.value_counts(normalize=True) * 100
test_pct = y_test.value_counts(normalize=True) * 100

comparison = pd.DataFrame({
    'Class': ['<=50K (0)', '>50K (1)'],
    'Full Dataset %': [class_pct_clean['<=50K'], class_pct_clean['>50K']],
    'Train %': [train_pct[0], train_pct[1]],
    'Test %':  [test_pct[0], test_pct[1]],
}).round(2)

print("📊 Class balance across full dataset, train, and test:")
print(comparison.to_string(index=False))
print("\n✅ Stratification preserved the class ratio across all three sets!")

In [ ]:
# ============================================================
# CELL 3-D: Build the preprocessing pipeline
# ============================================================
# A ColumnTransformer applies DIFFERENT preprocessing to different column types:
#   - Numeric columns   → StandardScaler (mean=0, std=1) — needed for SVC
#   - Categorical columns → OneHotEncoder — converts categories to 0/1 columns
#
# WHY ONE-HOT ENCODING?
# Models can't use text directly. OneHotEncoder turns a column like 'sex'
# (values: 'Male', 'Female') into two columns: 'sex_Male' (0/1) and 'sex_Female' (0/1).
#
# handle_unknown='ignore' → if the test set has a category not seen in training
# (rare, but possible with 'native-country'), it's encoded as all zeros instead of crashing.
#
# CRITICAL RULE (same as regression notebook): fit the preprocessor on TRAINING data only.

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)
])

# Quick fit-transform check on a sample
X_train_transformed_sample = preprocessor.fit_transform(X_train)
print(f"✅ Preprocessor fitted on training data.")
print(f"   Input features:  {X_train.shape[1]}")
print(f"   Output features after encoding: {X_train_transformed_sample.shape[1]}")
print("   (one-hot encoding expands each categorical column into multiple 0/1 columns)")

---
## 4. Cross-Validation

### 🔑 Stratified K-Fold for Classification
> Just like the train/test split, cross-validation folds should also preserve class proportions. **`StratifiedKFold`** does this automatically — each of the 5 folds has roughly the same `>50K` / `<=50K` ratio.

### 🤖 The 4 Algorithms
| Algorithm | How it works | Strengths |
|---|---|---|
| **SVC** | Finds the best boundary (hyperplane) separating classes, possibly in a higher-dimensional space (via kernels) | Effective with clear margins; sensitive to scaling |
| **Random Forest** | Ensemble of decision trees, majority vote | Robust, handles non-linearity, less prone to overfitting |
| **XGBoost** | Sequentially built trees that correct previous errors (gradient boosting) | Often state-of-the-art accuracy, handles missing data |
| **LightGBM** | Like XGBoost but grows trees leaf-wise; very fast on large data | Speed + accuracy, great for big datasets |

### 📏 Classification Metrics
| Metric | What it measures |
|---|---|
| **Accuracy** | % of predictions that were correct overall |
| **Precision** | Of everyone predicted `>50K`, how many actually were? |
| **Recall** | Of everyone actually `>50K`, how many did we catch? |
| **F1** | Harmonic mean of precision and recall (balances both) |
| **ROC-AUC** | How well the model ranks positives above negatives (0.5=random, 1.0=perfect) |

In [ ]:
# ============================================================
# CELL 4-A: Define all models (wrapped in preprocessing pipelines)
# ============================================================
# Each model is wrapped in a Pipeline with the SAME preprocessor.
# This guarantees identical preprocessing across all 4 algorithms,
# and ensures preprocessing is refit correctly on each CV fold (no data leakage).

def make_pipeline(model):
    return Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

models = {
    'SVC':            make_pipeline(SVC(probability=True, random_state=RANDOM_STATE)),
    'Random Forest':  make_pipeline(RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=RANDOM_STATE)),
    'XGBoost':        make_pipeline(XGBClassifier(n_estimators=100, max_depth=4, eval_metric='logloss', random_state=RANDOM_STATE)),
    'LightGBM':       make_pipeline(LGBMClassifier(n_estimators=100, max_depth=-1, random_state=RANDOM_STATE, verbose=-1)),
}

print("🤖 Models registered (each wrapped in a preprocessing pipeline):")
for name in models:
    print(f"   • {name}")

In [ ]:
# ============================================================
# CELL 4-B: Run 5-fold Stratified Cross-Validation
# ============================================================
# NOTE: SVC can be slow on ~26K training rows with default settings.
# We use a SAMPLE of the training data for SVC's cross-validation to keep
# runtime manageable, while the tree models use the full training set.
# (The final tuned models in Section 5/6 will still be evaluated fairly on the full test set.)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = {}

print("Running 5-fold stratified cross-validation...")
print("(SVC may take a few minutes — using a 8,000-row sample for CV speed)\n")

# Sample for SVC to keep cross-validation fast
sample_idx = X_train.sample(8000, random_state=RANDOM_STATE).index
X_train_sample = X_train.loc[sample_idx]
y_train_sample = y_train.loc[sample_idx]

for name, model in models.items():
    print(f"  Evaluating: {name}...", end=' ', flush=True)

    if name == 'SVC':
        X_cv, y_cv = X_train_sample, y_train_sample
    else:
        X_cv, y_cv = X_train, y_train

    scores = cross_validate(model, X_cv, y_cv, cv=cv, scoring=scoring, n_jobs=-1)

    cv_results[name] = {f'{m}_mean': scores[f'test_{m}'].mean() for m in scoring}
    cv_results[name].update({f'{m}_std': scores[f'test_{m}'].std() for m in scoring})

    print(f"Accuracy={cv_results[name]['accuracy_mean']:.4f}, F1={cv_results[name]['f1_mean']:.4f}")

cv_df = pd.DataFrame(cv_results).T.round(4)
print("\n📊 Cross-Validation Results Summary:")
print(cv_df[['accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'roc_auc_mean']].to_string())
print("\n⚠️  Note: SVC was evaluated on an 8,000-row sample; other models on the full training set.")
print("    Final comparisons in Section 6 use the same held-out test set for all models.")

In [ ]:
# ============================================================
# CELL 4-C: Visualize cross-validation results
# ============================================================

model_names = list(cv_results.keys())
metrics_to_plot = ['accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'roc_auc_mean']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_to_plot))
width = 0.2

for i, name in enumerate(model_names):
    values = [cv_results[name][m] for m in metrics_to_plot]
    ax.bar(x + i*width, values, width, label=name, color=colors[i], alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Cross-Validation Model Comparison (5-Fold Stratified)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

best_model_name = max(cv_results, key=lambda m: cv_results[m]['f1_mean'])
print(f"🏆 Best model by CV F1: {best_model_name} (F1 = {cv_results[best_model_name]['f1_mean']:.4f})")

---
## 5. Hyperparameter Tuning

### 🔑 Key Hyperparameters for Each Algorithm

| Algorithm | Key Hyperparameters |
|---|---|
| **SVC** | `C` (regularization strength), `kernel` (linear/rbf), `gamma` (kernel coefficient) |
| **Random Forest** | `n_estimators`, `max_depth`, `min_samples_split`, `max_features` |
| **XGBoost** | `n_estimators`, `max_depth`, `learning_rate`, `subsample` |
| **LightGBM** | `n_estimators`, `num_leaves`, `learning_rate`, `subsample` |

Since `models` dict already wraps preprocessing + model in a Pipeline, we reference hyperparameters with the `model__` prefix (e.g., `model__C` for SVC's `C`).

In [ ]:
# ============================================================
# CELL 5-A: Tune SVC
# ============================================================
# SVC is the slowest of the 4 — we use a smaller training sample and
# a focused grid to keep runtime reasonable.

print("🔧 Tuning SVC (Grid Search on 8,000-row sample)...")

svc_param_grid = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['rbf', 'linear'],
    'model__gamma': ['scale', 'auto'],
}

svc_grid = GridSearchCV(
    make_pipeline(SVC(probability=True, random_state=RANDOM_STATE)),
    svc_param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
svc_grid.fit(X_train_sample, y_train_sample)

print(f"\nBest SVC parameters: {svc_grid.best_params_}")
print(f"Best CV F1: {svc_grid.best_score_:.4f}")

In [ ]:
# ============================================================
# CELL 5-B: Tune Random Forest
# ============================================================
print("🔧 Tuning Random Forest (Randomized Search)...")

rf_param_dist = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [8, 12, 16, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 0.5, 0.8],
}

rf_random = RandomizedSearchCV(
    make_pipeline(RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE)),
    rf_param_dist,
    n_iter=20,
    cv=3,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
rf_random.fit(X_train, y_train)

print(f"\nBest RF parameters: {rf_random.best_params_}")
print(f"Best CV F1: {rf_random.best_score_:.4f}")

In [ ]:
# ============================================================
# CELL 5-C: Tune XGBoost
# ============================================================
print("🔧 Tuning XGBoost (Randomized Search)...")

xgb_param_dist = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 4, 5, 6],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_random = RandomizedSearchCV(
    make_pipeline(XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE)),
    xgb_param_dist,
    n_iter=20,
    cv=3,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
xgb_random.fit(X_train, y_train)

print(f"\nBest XGBoost parameters: {xgb_random.best_params_}")
print(f"Best CV F1: {xgb_random.best_score_:.4f}")

In [ ]:
# ============================================================
# CELL 5-D: Tune LightGBM
# ============================================================
print("🔧 Tuning LightGBM (Randomized Search)...")

lgbm_param_dist = {
    'model__n_estimators': [100, 200, 300],
    'model__num_leaves': [15, 31, 63, 127],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
}

lgbm_random = RandomizedSearchCV(
    make_pipeline(LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)),
    lgbm_param_dist,
    n_iter=20,
    cv=3,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
lgbm_random.fit(X_train, y_train)

print(f"\nBest LightGBM parameters: {lgbm_random.best_params_}")
print(f"Best CV F1: {lgbm_random.best_score_:.4f}")

---
## 6. Results Validation

Now we evaluate all 4 tuned models on the **held-out test set** — data none of them have seen during training or tuning.

In [ ]:
# ============================================================
# CELL 6-A: Evaluate all tuned models on the test set
# ============================================================

tuned_models = {
    'SVC (tuned)':            svc_grid.best_estimator_,
    'Random Forest (tuned)':  rf_random.best_estimator_,
    'XGBoost (tuned)':        xgb_random.best_estimator_,
    'LightGBM (tuned)':       lgbm_random.best_estimator_,
}

test_results = {}
print("📊 Test Set Performance (final evaluation):")
print(f"{'Model':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>8}")
print("-" * 75)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    test_results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': auc,
                           'y_pred': y_pred, 'y_proba': y_proba}
    print(f"{name:<25} {acc:>9.4f} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f} {auc:>8.4f}")

best_name = max(test_results, key=lambda m: test_results[m]['f1'])
print(f"\n🏆 Best model on test set: {best_name} (F1={test_results[best_name]['f1']:.4f})")

In [ ]:
# ============================================================
# CELL 6-B: Confusion matrices
# ============================================================
# A CONFUSION MATRIX shows 4 numbers:
#   - True Negatives (TN):  correctly predicted <=50K
#   - False Positives (FP): predicted >50K but actually <=50K
#   - False Negatives (FN): predicted <=50K but actually >50K
#   - True Positives (TP):  correctly predicted >50K
#
# Precision = TP / (TP + FP)  — 'of those I said >50K, how many were right?'
# Recall    = TP / (TP + FN)  — 'of everyone actually >50K, how many did I catch?'

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, (name, result) in enumerate(test_results.items()):
    cm = confusion_matrix(y_test, result['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K'])
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f"{name}\nF1={result['f1']:.3f}", fontsize=10)

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6-C: ROC curves
# ============================================================
# The ROC CURVE plots True Positive Rate vs False Positive Rate at every
# possible decision threshold. A model that's purely guessing follows the
# diagonal line (AUC=0.5). A perfect model hugs the top-left corner (AUC=1.0).

plt.figure(figsize=(8, 8))

for name, result in test_results.items():
    fpr, tpr, _ = roc_curve(y_test, result['y_proba'])
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={result['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random guess (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models (Test Set)', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6-D: Detailed classification report for the best model
# ============================================================
# precision/recall/F1 PER CLASS — useful to see if the model is biased
# toward the majority class (<=50K).

print(f"📋 Classification Report — {best_name}\n")
print(classification_report(y_test, test_results[best_name]['y_pred'],
                             target_names=['<=50K', '>50K']))

In [ ]:
# ============================================================
# CELL 6-E: Feature importance
# ============================================================
# We extract feature names AFTER one-hot encoding from the preprocessor,
# then pull built-in importance scores from the tree-based models.

# Get the expanded feature names from the fitted preprocessor
ohe = tuned_models['Random Forest (tuned)'].named_steps['preprocess'].named_transformers_['cat']
ohe_feature_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = NUMERIC_FEATURES + list(ohe_feature_names)

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

tree_models = {'Random Forest (tuned)': axes[0], 'XGBoost (tuned)': axes[1], 'LightGBM (tuned)': axes[2]}

for name, ax in tree_models.items():
    model = tuned_models[name].named_steps['model']
    importances = pd.Series(model.feature_importances_, index=all_feature_names).sort_values(ascending=False)
    top15 = importances.head(15)

    ax.barh(top15.index[::-1], top15.values[::-1], color='steelblue', alpha=0.85)
    ax.set_title(f'{name}: Top 15 Features', fontsize=11)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Map top features back to original column descriptions where possible
rf_importances = pd.Series(
    tuned_models['Random Forest (tuned)'].named_steps['model'].feature_importances_,
    index=all_feature_names
).sort_values(ascending=False)

print("💡 Top 10 most important features (Random Forest):")
for feat, imp in rf_importances.head(10).items():
    # Map encoded feature back to original column name for description lookup
    matched = next((c for c in FEATURE_DESCRIPTIONS if feat.startswith(c)), None)
    desc = describe(matched) if matched else 'Encoded category'
    bar = '█' * int(imp * 100)
    print(f"  {feat:<30} {imp:.4f}  {bar}")
    print(f"      \u2514\u2500 {desc}")

In [ ]:
# ============================================================
# CELL 6-F: Overfitting check — Train vs Test performance
# ============================================================
print("📊 Overfitting Check: Train vs Test F1")
print(f"{'Model':<25} {'Train F1':>10} {'Test F1':>10} {'Overfit?':>10}")
print("-" * 60)

for name, model in tuned_models.items():
    if name == 'SVC (tuned)':
        train_pred = model.predict(X_train_sample)
        train_f1 = f1_score(y_train_sample, train_pred)
    else:
        train_pred = model.predict(X_train)
        train_f1 = f1_score(y_train, train_pred)

    test_f1 = test_results[name]['f1']
    gap = train_f1 - test_f1
    flag = '⚠️ Yes' if gap > 0.05 else '✅ No'
    print(f"{name:<25} {train_f1:>10.4f} {test_f1:>10.4f} {flag:>10}")

print("\n(SVC's train F1 is computed on its 8,000-row training sample for consistency with CV)")

In [ ]:
# ============================================================
# CELL 6-G: Final summary dashboard
# ============================================================

print("\n" + "="*70)
print("  CLASSIFICATION ANALYSIS — FINAL SUMMARY")
print("="*70)

print("""
DATASET
  - Adult / Census Income (UCI)
  - ~45,000 rows after dropping missing values, 12 predictive features
  - Target: income > $50K (binary)

PREPROCESSING
  - Dropped rows with missing workclass/occupation/native-country ('?' values)
  - Dropped 'fnlwgt' (sampling weight) and 'education-num' (redundant)
  - StandardScaler for numeric features, OneHotEncoder for categoricals
  - All wrapped in a single sklearn Pipeline per model

TRAIN/TEST SPLIT
  - Stratified 80/20 split (preserves ~76%/24% class ratio)
  - Reproducible using random_state=42

ASSUMPTIONS VALIDATION
  - Class imbalance: ~3:1 ratio → tracked precision/recall/F1/ROC-AUC, not just accuracy
  - Multicollinearity: education/education-num redundancy resolved by dropping one
  - Feature scaling: required for SVC, harmless for tree models
  - High cardinality: native-country handled via OneHotEncoder(handle_unknown='ignore')

CROSS-VALIDATION (5-fold stratified)
  - Tree-based models (RF, XGBoost, LightGBM) generally outperform SVC on F1/ROC-AUC
  - Gradient boosting models capture non-linear feature interactions effectively
""")

print("MODEL COMPARISON (Test Set):")
print(f"{'Model':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>8}")
print("-" * 75)
for name, result in sorted(test_results.items(), key=lambda x: -x[1]['f1']):
    star = ' ← BEST' if name == best_name else ''
    print(f"{name:<25} {result['accuracy']:>9.4f} {result['precision']:>10.4f} "
          f"{result['recall']:>8.4f} {result['f1']:>8.4f} {result['roc_auc']:>8.4f}{star}")

print(f"""
KEY INSIGHTS (from feature importance)
  1. Capital gain/loss are strong predictors despite most values being $0
  2. Age and hours-per-week are consistently important
  3. Marital status and relationship status carry strong signal
  4. Education level strongly correlates with income

NEXT STEPS
  → Run this notebook on cloud platforms (GCP, Azure, AWS) alongside the regression notebook
  → Part 2: Wrap best model as a FastAPI endpoint
  → Part 3: Build Streamlit/Gradio demo
  → Consider: SMOTE or class_weight tuning for the imbalanced target
""")

print("="*70)
print("✅ Analysis complete!")
print("="*70)

---
## 📚 Concepts Glossary

| Term | Plain English Definition |
|---|---|
| **Classification** | Predicting a category/label (vs. a continuous number) |
| **Binary Classification** | Classification with exactly 2 possible labels |
| **Class Imbalance** | One class is much more frequent than the other |
| **Stratified Split** | Train/test split that preserves class proportions |
| **One-Hot Encoding** | Converts a category into multiple 0/1 columns |
| **ColumnTransformer** | Applies different preprocessing to different columns |
| **Confusion Matrix** | Table of TP/TN/FP/FN — shows where predictions go right or wrong |
| **Precision** | Of predicted positives, how many were actually positive? |
| **Recall (Sensitivity)** | Of actual positives, how many did we correctly identify? |
| **F1 Score** | Harmonic mean of precision and recall |
| **ROC Curve** | Plot of true positive rate vs false positive rate across thresholds |
| **ROC-AUC** | Area under the ROC curve; overall ranking quality (0.5=random, 1.0=perfect) |
| **SVC** | Support Vector Classifier — finds the best separating boundary between classes |
| **Random Forest** | Ensemble of decision trees voting on the final prediction |
| **XGBoost / LightGBM** | Gradient boosting — trees built sequentially, each correcting prior errors |
| **Cardinality** | Number of unique values in a categorical column |
| **Pipeline** | Chains preprocessing + model into one object, prevents data leakage |